In [ ]:
import os
os.environ["GROQ_API_KEY"] = "**"

In [ ]:
!pip install -q requests langchain langchain-core langchain_groq

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool, InjectedToolArg
from langchain_core.messages import HumanMessage
import requests
from typing import Annotated

In [ ]:
# create tool

@tool
def get_factor(base_currency: str, target_currency: str) -> float:
  """this function fetches currency conversion factor between a given base currency and target currency"""

  url = f"https://v6.exchangerate-api.com/v6/db8649d92c90815c653f4262/pair/{base_currency}/{target_currency}"
  response = requests.get(url)
  data = response.json()
  return data

In [ ]:
get_factor.invoke({"base_currency":"USD", "target_currency":"INR"})

In [ ]:
@tool
def convert(base_currency_value:int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """given a currency conversion rate this function calculates the target currency value from a given base currency value"""
  return base_currency_value*conversion_rate

In [ ]:
convert.args

In [ ]:
convert.invoke({"base_currency_value":10, "conversion_rate":95.16})

In [ ]:
# tool binding

llm = ChatGroq(model="llama-3.1-8b-instant")

In [ ]:
llm_tools = llm.bind_tools([get_factor, convert])

In [ ]:
messages = [HumanMessage('What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr')]

In [ ]:
messages

In [ ]:
ai_msg = llm_tools.invoke(messages)

In [ ]:
messages.append(ai_msg)

In [ ]:
ai_msg

In [ ]:
ai_msg.tool_calls

In [ ]:
import json

for call in ai_msg.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if call['name'] == "get_factor":
    tool_msg1 = get_factor.invoke(call)
    # print(tool_msg1)

  # fetch conversion rate
  rate = json.loads(tool_msg1.content)['conversion_rate']

  messages.append(tool_msg1)

  # execute 2nd tool
  if call['name'] == "convert":
    call['args']['conversion_rate'] = rate
    tool_msg2 = convert.invoke(call)
    messages.append(tool_msg2)

In [ ]:
messages

In [ ]:
llm_tools.invoke(messages).content